In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error,root_mean_squared_error

In [ ]:
df = pd.read_excel("data/datos_limpios.xlsx")
df

In [ ]:
df.columns.unique()

CREACION DEL MODELO INICIAL

In [ ]:
from sklearn.model_selection import train_test_split

X = df[['Edad', 'Ingresos', 'Monto_Inicial', 'Scoring_Crediticio',
       'Meses_Empleo', 'Num_Creditos', 'Ratio_Interes', 'Duracion',
       'Ratio_Deuda_Ingresos', 'Estudios', 'Posesion_Hipoteca',
       'Personas_Cargo', 'Fiador', 'Impago', 'Estado_Civil_Casado',
       'Estado_Civil_Divorciado', 'Estado_Civil_Soltero',
       'Tipo_Jornada_Laboral_Autónomo', 'Tipo_Jornada_Laboral_Desempleado',
       'Tipo_Jornada_Laboral_Jornada completa',
       'Tipo_Jornada_Laboral_Tiempo parcial']] 
y = df[["Prima"]]

X_train, X_test, y_train, y_test = train_test_split(X, y,train_size = 0.8,random_state = 42,shuffle = True)

In [ ]:
# 1. Instanciamos el escalador
escalador = StandardScaler()

# 2. "Entrenamos" el escalador SOLO con los datos de Train y los transformamos
# Usamos un DataFrame de Pandas para no perder los nombres de las columnas
X_train_scaled = pd.DataFrame(
    escalador.fit_transform(X_train), 
    columns=X_train.columns, 
    index=X_train.index
)

# 3. Transformamos los datos de Test usando la misma regla que aprendió del Train
X_test_scaled = pd.DataFrame(
    escalador.transform(X_test), 
    columns=X_test.columns, 
    index=X_test.index
)

In [ ]:
# Creamos la instancia del modelo
modelo_sklearn = LinearRegression()

# Entrenamos el modelo asegurándonos de convertir y_train a un formato unidimensional con squeeze()
modelo_sklearn.fit(X=X_train_scaled, y=y_train.squeeze())

print("Intercepto (Productividad base estandarizada):", round(modelo_sklearn.intercept_, 2))
print("Impacto de cada variable (en desviaciones estandar):")

# Iteramos juntando los nombres originales con los coeficientes generados
for nombre, coeficiente in zip(X.columns, modelo_sklearn.coef_):
    print(f" - {nombre}: {round(coeficiente, 2)}")

print(f"Coeficiente de determinación R^2:", round(modelo_sklearn.score(X_test_scaled, y_test), 2))

In [ ]:
# 1. Hacemos las predicciones sobre el X_test escalado
predicciones = modelo_sklearn.predict(X=X_test_scaled)

# 2. Calculamos los errores
mse = mean_squared_error(y_true=y_test, y_pred=predicciones)
rmse = root_mean_squared_error(y_true=y_test, y_pred=predicciones)
mae = mean_absolute_error(y_true=y_test, y_pred=predicciones)

print("\n--- Resultados de la Evaluación ---")
print("RMSE (Raíz del Error Cuadrático Medio):", round(rmse, 2))
print("MAE (Error Absoluto Medio):", round(mae, 2))


In [ ]:
# 1. Añadimos la constante a los datos escalados de entrenamiento
X_train_sm = sm.add_constant(X_train_scaled, prepend=True)

# 2. Creamos y entrenamos el modelo estadístico
modelo_sm = sm.OLS(endog=y_train, exog=X_train_sm)
resultados_sm = modelo_sm.fit()

# 3. Imprimimos el resumen
print(resultados_sm.summary())

In [ ]:
# 1. Extraemos predicciones y sus bandas de confianza
predicciones_sm = resultados_sm.get_prediction(exog=X_train_sm).summary_frame(alpha=0.05)

# 2. Añadimos el valor real y ordenamos para el dibujo
predicciones_sm["y_real"] = y_train.values
predicciones_sm = predicciones_sm.sort_values("mean")

# 3. Creamos el lienzo
fig, ax = plt.subplots(figsize=(10, 7))

# 4. Puntos de Dispersión
ax.scatter(predicciones_sm["mean"], predicciones_sm["y_real"], marker="o", color="gray", alpha=0.5)

# 5. Línea ideal del modelo
ax.plot(predicciones_sm["mean"], predicciones_sm["mean"], linestyle="-", color="red", label="OLS")

ax.plot(predicciones_sm["mean"], predicciones_sm["mean_ci_lower"], linestyle="--", color="blue", label="95% CI")
ax.plot(predicciones_sm["mean"], predicciones_sm["mean_ci_upper"], linestyle="--", color="blue")
ax.fill_between(predicciones_sm["mean"], predicciones_sm["mean_ci_lower"], predicciones_sm["mean_ci_upper"], alpha=0.2, color="blue")

ax.plot(predicciones_sm["mean"], predicciones_sm["obs_ci_lower"], linestyle=":", color="orange", label="95% CI")
ax.plot(predicciones_sm["mean"], predicciones_sm["obs_ci_upper"], linestyle=":", color="orange")

ax.legend()

plt.show()

CREACION DEL MODELO AJUSTADO, POLINOMICO

In [ ]:
# 2. Seleccionar variables (omitimos las redundantes y no significativas del análisis previo)
variables_predictoras = ['Edad', 'Ingresos', 'Monto_Inicial', 'Scoring_Crediticio',
    'Ratio_Interes', 'Duracion', 'Ratio_Deuda_Ingresos', 
    'Estudios', 'Estado_Civil_Casado']

X = df[variables_predictoras]
y = df['Prima']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y,train_size = 0.8,random_state = 42,shuffle = True)

In [ ]:
# Generamos la clase polinómica
poly_features = PolynomialFeatures(degree=3, include_bias=False)

# Transformamos los datos de entrenamiento y prueba
X_train_poly = poly_features.fit_transform(X_train)
X_test_poly = poly_features.transform(X_test)

# Mantenemos los nombres de las variables generadas
nombres_poly = poly_features.get_feature_names_out(variables_predictoras)
X_train_poly = pd.DataFrame(X_train_poly, columns=nombres_poly, index=X_train.index)
X_test_poly = pd.DataFrame(X_test_poly, columns=nombres_poly, index=X_test.index)

# Estandarizamos para estabilizar el cálculo
escalador = StandardScaler()
X_train_scaled = pd.DataFrame(escalador.fit_transform(X_train_poly), columns=nombres_poly, index=X_train.index)
X_test_scaled = pd.DataFrame(escalador.transform(X_test_poly), columns=nombres_poly, index=X_test.index)

In [ ]:
# Añadimos la constante
X_train_sm = sm.add_constant(X_train_scaled, prepend=True)
X_test_sm = sm.add_constant(X_test_scaled, prepend=True)

# Creamos y entrenamos el modelo estadístico
modelo_sm = sm.OLS(endog=y_train, exog=X_train_sm)
resultados_sm = modelo_sm.fit()

# Imprimimos el resumen
print(resultados_sm.summary())

# Intervalos de confianza al 95%
intervalos_ci = resultados_sm.conf_int(alpha=0.05)
intervalos_ci.columns = ['2.5%', '97.5%']
print("\nIntervalos de Confianza:\n", intervalos_ci.head())

In [ ]:
# Validación Cruzada
modelo_cv = LinearRegression()
mse_scores = cross_val_score(modelo_cv, X_train_scaled, y_train, cv=5, scoring='neg_mean_squared_error')
rmse_cv = np.sqrt(-mse_scores.mean())
print(f"RMSE Promedio Estimado (CV-5): {rmse_cv:.2f}")

# Generamos la curva de aprendizaje
train_sizes, train_scores, valid_scores = learning_curve(
    LinearRegression(), X_train_scaled, y_train, 
    train_sizes=np.linspace(0.1, 1.0, 50),
    cv=5, scoring="neg_root_mean_squared_error"
)

# Promediamos y corregimos el signo negativo
train_errors = -train_scores.mean(axis=1)
valid_errors = -valid_scores.mean(axis=1)

# Gráfico de Curva de Aprendizaje
plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_errors, "r-+", linewidth=2, label="Entrenamiento")
plt.plot(train_sizes, valid_errors, "b-", linewidth=3, label="Validación")
plt.legend()
plt.xlabel("Tamaño del conjunto de entrenamiento")
plt.ylabel("RMSE")
plt.title("Curva de Aprendizaje: Diagnóstico de Sesgo y Varianza")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Generamos predicciones en Test
predicciones_test = resultados_sm.predict(X_test_sm)

# Errores globales
rmse_final = np.sqrt(mean_squared_error(y_test, predicciones_test))
mae_final = mean_absolute_error(y_test, predicciones_test)
print(f"\nRMSE de Test: {rmse_final:.2f}")
print(f"MAE de Test: {mae_final:.2f}")

# Diagnóstico de residuos en Entrenamiento
prediccion_train = resultados_sm.predict(X_train_sm)
residuos_train = prediccion_train - y_train.squeeze()

# Test formales de Normalidad
shapiro_test = stats.shapiro(residuos_train[:5000]) # Se limita por rendimiento
k2, p_value = stats.normaltest(residuos_train)
print(f"\nTest D'Agostino's K-squared: p-value = {p_value:.4f}")

# --- Construcción del Gráfico de Diagnóstico Múltiple ---
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 10))

# 1. Valor predicho vs Valor real
axes[0, 0].scatter(y_train, prediccion_train, edgecolors=(0, 0, 0), alpha=0.4, color='steelblue')
axes[0, 0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'k--', lw=2, color='yellow')
axes[0, 0].set_title('Predicho vs Real')
axes[0, 0].set_xlabel('Real')
axes[0, 0].set_ylabel('Predicción')

# 2. Residuos en el tiempo / índice (Detectar autocorrelación)
axes[0, 1].scatter(range(len(y_train)), residuos_train, edgecolors=(0, 0, 0), alpha=0.4, color='mediumseagreen')
axes[0, 1].axhline(y=0, linestyle='--', color='yellow', lw=2)
axes[0, 1].set_title('Residuos secuenciales')
axes[0, 1].set_xlabel('ID')
axes[0, 1].set_ylabel('Residuo')

# 3. Histograma de Residuos (Normalidad)
sns.histplot(data=residuos_train, stat="density", kde=True, line_kws={'linewidth': 1}, alpha=0.3, ax=axes[1, 0], color='purple')
axes[1, 0].set_title('Distribución de Residuos')
axes[1, 0].set_xlabel("Residuo")

# 4. Q-Q Plot (Distribución teórica Normal vs Real)
sm.qqplot(residuos_train, fit=True, line='q', ax=axes[1, 1], alpha=0.4, lw=2)
axes[1, 1].set_title('Q-Q plot de Residuos')

fig.tight_layout()
plt.show()

In [ ]:
# Asumimos que ya tienes calculada la variable 'residuos' tras hacer: 
residuos = y_test - predicciones

print("--- TESTS FORMALES DE NORMALIDAD DE LOS RESIDUOS ---")

# 1. Test de Shapiro-Wilk
# Nota: Shapiro-Wilk puede ser muy sensible en muestras gigantes (>5000),
# por lo que a veces es útil pasarle una muestra de los residuos.
shapiro_stat, shapiro_p = stats.shapiro(residuos)
print(f"Test de Shapiro-Wilk: Estadístico = {shapiro_stat:.4f}, p-value = {shapiro_p:.4e}")

# 2. Test K-cuadrado de D'Agostino
# Evalúa qué tanto se desvía la distribución de la asimetría y curtosis esperada.
k2_stat, k2_p = stats.normaltest(residuos)
print(f"Test de D'Agostino K-squared: Estadístico = {k2_stat:.4f}, p-value = {k2_p:.4e}")

# Interpretación general para que se imprima junto con los resultados
print("\nInterpretación:")
print("Si el p-value es < 0.05, rechazamos la hipótesis nula: Los residuos NO siguen una distribución normal.")
print("Si el p-value es >= 0.05, no hay evidencia para descartar la normalidad.")